# Stage B3 — Retrieval evaluation (the real test)

**Experiment B — Practical application: MobileCLIP → SigLIP 2 adapter**

Goal: one linear matrix that maps iPhone-tier MobileCLIP-S1 image
embeddings into server-tier SigLIP 2 space, so a **single Qdrant
collection** serves both tiers — the phone indexes images offline, the
server queries the same index with SigLIP text embeddings.


## What this stage does
Text-to-image retrieval on held-out images. The query is a SigLIP **text**
embedding (exactly what the server does in production). Three galleries:
- **A. SigLIP native** image embeddings → the ceiling
- **B. MobileCLIP + adapter** → our system
- **C. MobileCLIP raw** (no adapter) → lower baseline, expected ≈ 0

This is the criterion that separates "statistical correlation" from "the
information actually transfers": R² can be decent while the nuances that
drive retrieval are lost.

## Success criterion
Variant B achieves **≥ 90%** of variant A's Recall@1/5/10.
If 70-90%: upgrade the adapter to a small 1-2 layer MLP and re-run.
If < 70%: keep separate indexes per tier.


In [ ]:
# Storage setup — where stage artifacts (.npz, .png) are read/written.
# Each stage reads the previous stage's output from DATA_DIR.
#
# Option 1 (default): current directory. Works if you run ALL stages in
# the SAME runtime/session. In Colab, a new notebook = a new VM, so files
# from a previous notebook are gone.
#
# Option 2 (Colab, persistent): mount Google Drive and point DATA_DIR
# there — artifacts survive across notebooks and sessions:
#
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

import os
os.environ.setdefault("DATA_DIR", ".")
print("DATA_DIR =", os.path.abspath(os.environ["DATA_DIR"]))

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Functions
def recall_at_k(sim, ks=(1, 5, 10)):
    """sim: [n_queries, n_gallery], ground truth is the diagonal."""
    ranks = (-sim).argsort(axis=1)
    n = sim.shape[0]
    out = {}
    for k in ks:
        hits = (ranks[:, :k] == np.arange(n)[:, None]).any(1).mean()
        out[k] = hits
    return out


def l2n(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)


def main():
    pairs = np.load(str(DATA_DIR / "pairs.npz"))
    ad = np.load(str(DATA_DIR / "adapter.npz"))
    te = ad["eval_idx"]

    txt = pairs["sig_txt"][te]           # queries (server side)
    sig_img = pairs["sig_img"][te]       # ceiling gallery
    mob_img = pairs["mob_img"][te]

    W = ad["W_ridge"]
    adapted = l2n(mob_img @ W)           # our system gallery

    # pad raw mobileclip to siglip dim for the (expected-to-fail) baseline
    d_sig = sig_img.shape[1]
    raw = mob_img
    if raw.shape[1] < d_sig:
        raw = np.pad(raw, ((0, 0), (0, d_sig - raw.shape[1])))
    raw = l2n(raw[:, :d_sig])

    results = {
        "A. SigLIP native (ceiling)": recall_at_k(txt @ sig_img.T),
        "B. MobileCLIP + adapter   ": recall_at_k(txt @ adapted.T),
        "C. MobileCLIP raw (base)  ": recall_at_k(txt @ raw.T),
    }

    print(f"Held-out gallery size: {len(te)} images\n")
    print(f"{'variant':<30} R@1     R@5     R@10")
    for name, r in results.items():
        print(f"{name:<30} {r[1]:.3f}   {r[5]:.3f}   {r[10]:.3f}")

    ceil = results["A. SigLIP native (ceiling)"]
    ours = results["B. MobileCLIP + adapter   "]
    for k in (1, 5, 10):
        pct = 100 * ours[k] / max(ceil[k], 1e-9)
        verdict = "PASS" if pct >= 90 else "below target"
        print(f"\nR@{k}: adapter keeps {pct:.1f}% of ceiling -> {verdict}",
              end="")
    print()

In [ ]:
# Run the evaluation (requires pairs.npz and adapter.npz)
main()